# 07 – AI Agent điều tra so với giải thích một bước

**Câu hỏi:** cho LLM tự lập kế hoạch và tra cứu nhiều bước (agent) có cho hồ sơ kiểm tra tốt hơn việc giải thích một bước từ hồ sơ tính sẵn không, và phải trả giá bao nhiêu?

**Thiết lập**
- Cùng hồ sơ, cùng model (`openai/gpt-oss-20b` trên Groq), cùng các điều quy chế bắt buộc.
- **Một bước** (`src/llm.py::explain_case`): LLM nhận hồ sơ tính sẵn (rule, mô hình bất thường, ngân sách) và các điều quy chế, rồi viết giải thích.
- **Agent** (`src/agent.py`): LLM lập kế hoạch → gọi tối đa 8 công cụ chỉ đọc (giao dịch liên quan, lịch sử người chi, hồ sơ nhà cung cấp, ngân sách, quy chế, kết luận trước đây của kế toán) → viết hồ sơ → code kiểm chứng → sửa một vòng nếu có lỗi.
- 6 hồ sơ thuộc 5 loại tình huống, lấy từ mẫu của notebook 06. Notebook chỉ đọc kết quả đã lưu (`data/processed/llm_runs/agent_compare_gpt-oss-20b.jsonl`).

In [1]:
import sys, json, re
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.width', 180); pd.set_option('display.max_colwidth', 100)
all_rows = [json.loads(l) for l in open(ROOT / 'data/processed/llm_runs/agent_compare_gpt-oss-20b.jsonl', encoding='utf-8') if l.strip()]
rows = [r for r in all_rows if r.get('agent')]          # hồ sơ agent chạy xong
failed = [r for r in all_rows if not r.get('agent')]    # agent thất bại: vẫn giữ giải thích một bước
for r in failed:
    print(f"Agent thất bại ở hồ sơ {r['txn_id']} ({r['kind']}): {r['agent_error'][:160]}")
print(f'{len(all_rows)} hồ sơ; agent chạy xong {len(rows)}, thất bại {len(failed)}')
TXN = re.compile(r'\b\d{6}-\d{6}\b')

6 hồ sơ; agent chạy xong 6, thất bại 0


## 1. So sánh theo hồ sơ

In [2]:
def single_metrics(r):
    s, e = r['single'], r['single']['explanation']
    text = ' '.join([e['summary']] + [x['text'] for x in e['reasons']] + e['items_to_verify'] + e['next_steps'])
    others = set(TXN.findall(text)) - {r['txn_id']}
    return {'Giao dịch khác được dẫn': len(others), 'Số lý do / phát hiện': len(e['reasons']),
            'Có xét yếu tố giảm nghi ngờ': False, 'Mục cần kiểm tra': len(e['items_to_verify']),
            'Trích dẫn hợp lệ': len(e['citations']), 'Cảnh báo kiểm chứng': len(s['warnings']),
            'Công cụ đã dùng': 0, 'Lượt gọi LLM': 1,
            'Token': (s['usage']['prompt_tokens'] or 0) + (s['usage']['completion_tokens'] or 0),
            'Thời gian xử lý API (s)': s['usage'].get('api_time_s')}

def agent_metrics(r):
    a, rep = r['agent'], r['agent']['report']
    text = ' '.join([rep['summary']] + [f['text'] for f in rep['findings']] + rep['items_to_verify'])
    others = (set(TXN.findall(text)) | {i for f in rep['findings'] for i in f['evidence']}) - {r['txn_id']}
    return {'Giao dịch khác được dẫn': len(others), 'Số lý do / phát hiện': len(rep['findings']),
            'Có xét yếu tố giảm nghi ngờ': any(f['assessment'] == 'co_the_giai_thich' for f in rep['findings']),
            'Mục cần kiểm tra': len(rep['items_to_verify']), 'Trích dẫn hợp lệ': len(rep['citations']),
            'Cảnh báo kiểm chứng': len(a['warnings']), 'Công cụ đã dùng': len({t['tool'] for t in a['trace']}),
            'Lượt gọi LLM': a['usage']['llm_calls'], 'Token': a['usage']['prompt_tokens'] + a['usage']['completion_tokens'],
            'Thời gian xử lý API (s)': a['usage'].get('api_time_s'), 'Lần tự sửa': a['revisions'],
            'Đề xuất': rep['recommended_action']}

per_case = []
for r in rows:
    per_case.append({'txn_id': r['txn_id'], 'Loại': r['kind'], 'Cách': 'Một bước', **single_metrics(r)})
    per_case.append({'txn_id': r['txn_id'], 'Loại': r['kind'], 'Cách': 'Agent', **agent_metrics(r)})
P = pd.DataFrame(per_case)
P

,txn_id,Loại,Cách,Giao dịch khác được dẫn,Số lý do / phát hiện,Có xét yếu tố giảm nghi ngờ,Mục cần kiểm tra,Trích dẫn hợp lệ,Cảnh báo kiểm chứng,Công cụ đã dùng,Lượt gọi LLM,Token,Thời gian xử lý API (s),Lần tự sửa,Đề xuất
0,202607-031599,Cảnh báo cao (rule),Một bước,1,1,False,3,1,0,0,1,2645,1.266792,NaN,NaN
1,202607-031599,Cảnh báo cao (rule),Agent,3,3,True,4,4,0,6,9,25215,NaN,0.0,chuyen_phe_duyet
2,202606-010677,Cảnh báo cao (rule),Một bước,0,2,False,5,3,1,0,1,3994,2.031429,NaN,NaN
3,202606-010677,Cảnh báo cao (rule),Agent,8,4,True,3,4,0,6,9,30432,NaN,0.0,yeu_cau_bo_sung
4,202608-012960,Chỉ mô hình bất thường,Một bước,0,3,False,3,2,0,0,1,3734,1.994084,NaN,NaN
5,202608-012960,Chỉ mô hình bất thường,Agent,0,2,True,5,3,0,3,6,13206,NaN,0.0,yeu_cau_bo_sung
6,202606-009499,Cụm mua dồn,Một bước,0,3,False,6,1,1,0,1,3770,2.490087,NaN,NaN
7,202606-009499,Cụm mua dồn,Agent,10,2,False,5,3,0,4,8,21859,NaN,1.0,yeu_cau_bo_sung
8,202608-008438,Cần review (rule),Một bước,0,2,False,3,2,0,0,1,3064,1.559805,NaN,NaN
9,202608-008438,Cần review (rule),Agent,0,2,True,3,3,0,3,6,13055,3.420734,0.0,yeu_cau_bo_sung


## 2. Tổng hợp

In [3]:
num = ['Giao dịch khác được dẫn', 'Số lý do / phát hiện', 'Mục cần kiểm tra', 'Trích dẫn hợp lệ', 'Cảnh báo kiểm chứng',
       'Công cụ đã dùng', 'Lượt gọi LLM', 'Token', 'Thời gian xử lý API (s)']
S = P.groupby('Cách')[num].mean().T[['Một bước', 'Agent']]
S.loc['Tỷ lệ có xét yếu tố giảm nghi ngờ'] = P.groupby('Cách')['Có xét yếu tố giảm nghi ngờ'].mean()[['Một bước', 'Agent']]
S['Agent / Một bước'] = S['Agent'] / S['Một bước'].replace(0, np.nan)
S.round(2)

Cách,Một bước,Agent,Agent / Một bước
Giao dịch khác được dẫn,0.17,4.00,24.00
Số lý do / phát hiện,2.17,2.67,1.23
Mục cần kiểm tra,4.00,4.00,1.00
Trích dẫn hợp lệ,2.00,3.33,1.67
Cảnh báo kiểm chứng,0.33,0.00,0.00
Công cụ đã dùng,0.00,4.17,NaN
Lượt gọi LLM,1.00,7.50,7.50
Token,3459.50,20372.67,5.89
Thời gian xử lý API (s),1.90,3.42,1.80
Tỷ lệ có xét yếu tố giảm nghi ngờ,0.00,0.67,NaN


In [4]:
a = P[P['Cách'] == 'Agent']
print('Đề xuất của agent:', a['Đề xuất'].value_counts().to_dict())
print('Số hồ sơ agent phải tự sửa:', int((a['Lần tự sửa'] > 0).sum()), '/', len(a))
print('Số hồ sơ có thời gian xử lý API:', P.groupby('Cách')['Thời gian xử lý API (s)'].count().to_dict(),
      '(agent chỉ có số đo ở các hồ sơ chạy sau khi thêm; trung bình của agent vì vậy chỉ mang tính tham khảo)')
print('Công cụ được dùng nhiều nhất:', pd.Series([t['tool'] for r in rows for t in r['agent']['trace']]).value_counts().to_dict())

Đề xuất của agent: {'yeu_cau_bo_sung': 4, 'chuyen_phe_duyet': 2}
Số hồ sơ agent phải tự sửa: 1 / 6
Số hồ sơ có thời gian xử lý API: {'Agent': 2, 'Một bước': 6} (agent chỉ có số đo ở các hồ sơ chạy sau khi thêm; trung bình của agent vì vậy chỉ mang tính tham khảo)
Công cụ được dùng nhiều nhất: {'get_cardholder_history': 7, 'get_related_transactions': 6, 'get_merchant_profile': 6, 'check_budget': 3, 'search_policy': 2, 'get_past_decisions': 2}


**Đọc kết quả** (xem bảng ở trên):
- **Độ sâu điều tra:** agent dẫn ra các giao dịch liên quan mà cách một bước không có (giao dịch cặp chia nhỏ, giao dịch cùng ngày, lịch sử ở cùng cửa hàng), và đánh giá từng phát hiện là đáng lo ngại, có thể giải thích hay chưa đủ thông tin. Nhờ vậy kế toán thấy được cả yếu tố làm **giảm** nghi ngờ (nhà cung cấp quen, ngân sách còn dư), chứ không chỉ các lý do bị cảnh báo.
- **Chi phí:** agent tốn nhiều lượt gọi LLM và nhiều token hơn hẳn. Token và số lượt gọi là thước đo chi phí chính: thời gian thực tế của thí nghiệm chủ yếu là thời gian chờ hạn mức của bản miễn phí Groq (8.000 token/phút, 200.000 token/ngày) nên không được dùng để so sánh. "Thời gian xử lý API" là thời gian Groq báo cho từng lượt gọi, không gồm thời gian chờ; các hồ sơ agent chạy trước khi có số đo này để trống. Vì vậy trong ứng dụng, agent chạy nền và chỉ chạy khi kế toán yêu cầu; giải thích một bước vẫn là mặc định.
- **Kiểm soát:** mọi bằng chứng của agent nằm trong tập giao dịch mà công cụ đã trả về (có thể gồm cả phần kết quả bị cắt ngắn trước khi đưa cho LLM, nên chưa chắc agent đã đọc), và đề xuất nằm trong các thao tác hợp lệ (ép bằng schema). Bộ kiểm chứng của code buộc agent tự sửa khi có lỗi.

## 3. Ví dụ đặt cạnh nhau

In [5]:
def show(r):
    e, rep = r['single']['explanation'], r['agent']['report']
    lines = [f"### {r['txn_id']} – {r['kind']}", '', '**Một bước:** ' + e['summary'], '']
    lines += [f"- {x['text']}" for x in e['reasons']]
    lines += ['', '**Agent:** ' + rep['summary'], '']
    lines += [f"- ({f['assessment']}) {f['text']} — bằng chứng: {', '.join(f['evidence'])}" for f in rep['findings']]
    lines += ['', f"Agent đề xuất: **{rep['recommended_action']}** – {rep['recommendation_reason']}",
              '', 'Các bước: ' + '; '.join(f"{t['label']}" for t in r['agent']['trace'])]
    display(Markdown('\n'.join(lines)))
for r in rows[:2]:
    show(r)

### 202607-031599 – Cảnh báo cao (rule)

**Một bước:** Giao dịch 202607-031599 bị đánh dấu rủi ro cao do vi phạm quy định Hóa đơn trùng (Rule R03).

- Vi phạm quy định Hóa đơn trùng: giao dịch trùng người chi, cửa hàng và số tiền $150.00 với giao dịch 202607-031596 trong cùng ngày.

**Agent:** Giao dịch 202607-031599 tại GELLCO CLOTHING & SHOES có dấu hiệu trùng với giao dịch 202607-031596 cùng ngày, cùng người chi và cùng số tiền 150 USD, đồng thời không có ngân sách được phân bổ cho nhóm chi tiêu APPAREL_SPORTS trong tháng 2026‑07. Các bằng chứng từ hệ thống cho thấy người chi đã thực hiện nhiều giao dịch tại nhà cung cấp này trong cùng kỳ, nhưng mức chi trung bình và số tiền 150 USD nằm trong phạm vi bình thường. Tuy nhiên, quy tắc R03 (duplicate invoice) đã đánh dấu cảnh báo cao và việc thiếu ngân sách tạo ra rủi ro vượt ngân sách. Do đó, cần xác minh thêm bằng chứng và chuyển phê duyệt lên quản lý.

- (dang_lo_ngai) Giao dịch 202607-031599 có dấu hiệu trùng với giao dịch 202607-031596 cùng ngày, cùng người chi, cùng cửa hàng và cùng số tiền 150 USD, vi phạm quy tắc R03 (duplicate invoice). — bằng chứng: 202607-031599, 202607-031596
- (co_the_giai_thich) Không có ngân sách được phân bổ cho nhóm chi tiêu Trang phục, đồng phục & đồ thể thao trong tháng 2026-07, gây rủi ro vượt ngân sách. — bằng chứng: 202607-031599
- (co_the_giai_thich) Người chi đã thực hiện 25 giao dịch tại GELLCO trong cùng kỳ, nhưng mức chi trung bình và số tiền 150 USD nằm trong phạm vi bình thường, không vi phạm quy định. — bằng chứng: 202607-031599, 202607-031596, 202607-031600, 202607-031604

Agent đề xuất: **chuyen_phe_duyet** – Giao dịch có dấu hiệu trùng và không có ngân sách, cần xác minh thêm và chuyển phê duyệt.

Các bước: Xem giao dịch liên quan; Xem lịch sử người chi; Xem hồ sơ nhà cung cấp; Kiểm tra ngân sách; Tìm trong quy chế; Xem kết luận trước đây của kế toán

### 202606-010677 – Cảnh báo cao (rule)

**Một bước:** Giao dịch 202606-010677 thuộc nhóm Utilities, trị giá 1461.98 USD, nằm trong chuỗi 7 giao dịch cùng nhà cung cấp trong 3 ngày, mỗi giao dịch dưới 5000 USD, tổng 17,071.47 USD, vi phạm quy định chia nhỏ giao dịch (Điều 5). Ngoài ra, ngày 15/06/2026 có 28 giao dịch tại cùng nhà cung cấp, tổng 34,934.98 USD, tạo dấu hiệu mua dồn (Điều 26). Hệ thống đánh giá mức rủi ro Cảnh báo cao. Cần kiểm tra chứng từ, phê duyệt, và giải trình.

- Vi phạm quy định chia nhỏ giao dịch (Điều 5) vì cùng người chi thực hiện 7 giao dịch tại cùng nhà cung cấp trong 3 ngày, mỗi giao dịch dưới 5.000 USD, tổng 17.071.47 USD vượt hạn mức 5.000 USD.
- Dấu hiệu mua dồn: 28 giao dịch tại cùng nhà cung cấp trong ngày 15/06/2026, tổng 34.934.98 USD, gấp 14.4 lần mức thường, theo điều 26.

**Agent:** Giao dịch 202606-010677 tại OKCUTILITIES.COM bị cảnh báo cao do vi phạm quy định chia nhỏ, mua dồn dập và thiếu chứng từ chi tiết. Cần bổ sung tài liệu để xác định tính hợp lệ.

- (dang_lo_ngai) Giao dịch 202606-010677 cùng với 4 giao dịch khác trong 3 ngày tại OKCUTILITIES.COM, mỗi khoản dưới 5.000 USD, tổng vượt 5.000 USD, vi phạm quy định chia nhỏ. — bằng chứng: 202606-010677, 202606-010665, 202606-010666, 202606-010668, 202606-010669
- (dang_lo_ngai) 15 giao dịch trong ngày 15/06/2026 tại cùng nhà cung cấp, vượt nhịp mua thường, vi phạm quy định mua dồn dập. — bằng chứng: 202606-010677, 202606-010670, 202606-010671, 202606-010672, 202606-010673
- (dang_lo_ngai) Giao dịch được đánh giá cảnh báo cao do tổng giá trị vượt hạn mức và dấu hiệu bất thường. — bằng chứng: 202606-010677
- (co_the_giai_thich) Giao dịch chưa kèm hóa đơn chi tiết, cần bổ sung theo quy định. — bằng chứng: 202606-010677

Agent đề xuất: **yeu_cau_bo_sung** – Giao dịch vi phạm quy định chia nhỏ, mua dồn dập và thiếu chứng từ chi tiết, cần bổ sung tài liệu để xác định tính hợp lệ.

Các bước: Xem giao dịch liên quan; Xem lịch sử người chi; Xem hồ sơ nhà cung cấp; Kiểm tra ngân sách; Tìm trong quy chế; Xem kết luận trước đây của kế toán

## 4. Phiếu chấm điểm thủ công (hai cách đặt cạnh nhau, ẩn tên cách)

In [6]:
rng = np.random.default_rng(3)
out = ROOT / 'data' / 'labeling' / 'agent_rating.xlsx'
key = []
if not out.exists():
    sheet = []
    for r in rows:
        e, rep = r['single']['explanation'], r['agent']['report']
        single_txt = e['summary'] + '\n' + '\n'.join('- ' + x['text'] for x in e['reasons']) + '\nCần kiểm tra:\n' + '\n'.join('- ' + x for x in e['items_to_verify'])
        agent_txt = rep['summary'] + '\n' + '\n'.join(f"- {f['text']} (bằng chứng: {', '.join(f['evidence'])})" for f in rep['findings']) + '\nCần kiểm tra:\n' + '\n'.join('- ' + x for x in rep['items_to_verify'])
        flip = bool(rng.integers(2))
        A, B = (agent_txt, single_txt) if flip else (single_txt, agent_txt)
        key.append({'txn_id': r['txn_id'], 'A_la': 'agent' if flip else 'mot_buoc'})
        sheet.append({'txn_id': r['txn_id'], 'Loại': r['kind'], 'Phương án A': A, 'Phương án B': B,
                      'A – đúng với hồ sơ (1-5)': None, 'A – hữu ích (1-5)': None,
                      'B – đúng với hồ sơ (1-5)': None, 'B – hữu ích (1-5)': None, 'Chọn A hay B': None, 'Nhận xét': None})
    pd.DataFrame(sheet).to_excel(out, index=False)
    pd.DataFrame(key).to_csv(ROOT / 'data' / 'labeling' / 'agent_rating_key.csv', index=False)
    print('Đã tạo', out, '(khóa giải mã A/B lưu riêng ở agent_rating_key.csv)')
else:
    print('Đã có phiếu chấm, không ghi đè:', out)

Đã có phiếu chấm, không ghi đè: D:\Thesis\data\labeling\agent_rating.xlsx


## 4b. Các kiểu lỗi của agent đã quan sát

| Kiểu lỗi | Ví dụ | Xử lý |
|---|---|---|
| Coi lỗi tra cứu là bằng chứng | Agent gõ sai tên nhà cung cấp, công cụ báo không tìm thấy, agent viết "không có hồ sơ nhà cung cấp, nghi ngờ tính hợp pháp" | Sửa **trước** lần chạy này: hồ sơ có `merchant_norm`, công cụ gợi ý tên đúng, prompt nêu rõ lỗi công cụ không phải bằng chứng |
| Hiểu sai "không áp dụng" thành rủi ro | Không có ngân sách riêng cho nhóm chi phí → agent viết "gây rủi ro vượt ngân sách" | Quan sát **trong** lần chạy này; đã sửa sau bằng thông báo rõ nghĩa của công cụ ("không xét quy tắc ngân sách, không phải dấu hiệu rủi ro") |
| Số đếm không khớp nguồn | Agent viết "15 giao dịch trong ngày" trong khi mô hình cụm mua dồn ghi 28 | **Chưa** bị bộ kiểm chứng bắt (chỉ kiểm cặp số tiền – tỷ lệ); cần người chấm, hoặc mở rộng kiểm chứng sang số đếm |
| Báo cáo quá dài bị cắt | Cụm 28 giao dịch, agent liệt kê quá nhiều mã | Giới hạn độ dài đầu ra, yêu cầu viết ngắn hơn rồi thử lại |
| Request vượt 8.000 token (giới hạn Groq bản miễn phí) | Gửi lại toàn bộ hội thoại khi viết hồ sơ | Viết hồ sơ từ bản tóm lược bằng chứng có giới hạn độ dài |
| Model sinh lời gọi công cụ hỏng | Hồ sơ Amazon bình thường: Groq trả lỗi `output_parse_failed` ở vòng tra cứu trong 3 lần chạy liên tiếp (kể cả khi đã tự thử lại 2 lần), lần thứ 4 mới chạy xong | Gọi lại khi gặp lỗi này; nếu vẫn lỗi thì ghi nhận agent thất bại, hồ sơ vẫn có giải thích một bước. Với gpt-oss-20b, agent không ổn định bằng giải thích một bước |

Các lỗi này cho thấy **kiểm soát agent cần cả thiết kế công cụ** (thông báo rõ nghĩa, tham số chuẩn hóa), không chỉ prompt và schema.

## 5. Kết luận

1. **Agent điều tra sâu hơn:** tự tìm và dẫn ra các giao dịch liên quan, và cân nhắc cả yếu tố giảm nghi ngờ. Đây là những việc kế toán vẫn phải tự tra cứu khi chỉ có giải thích một bước.
2. **Chi phí cao hơn nhiều lần** (lượt gọi, token), nên agent là chức năng **theo yêu cầu**, chạy nền, không thay thế giải thích một bước.
3. **Kiểm soát bằng code vẫn là lớp chính:** bằng chứng và đề xuất bị giới hạn bằng schema, kết quả được kiểm chứng tự động, agent không thực hiện thao tác. Mức rủi ro và quyết định cuối cùng vẫn thuộc về code và con người.
4. **Hạn chế:** cỡ mẫu nhỏ (6 hồ sơ) do giới hạn bản miễn phí; chất lượng nội dung cần người chấm theo phiếu `data/labeling/agent_rating.xlsx` (ẩn tên phương án).